In [ ]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, precision_recall_curve, recall_score, f1_score, roc_curve, roc_auc_score

1.データの読み込み

In [ ]:
file_path = "../data/raw/train-00000-of-00001.parquet"
df_train_raw = pd.read_parquet(file_path) 

In [ ]:
file_path = "../data/raw/test-00000-of-00001.parquet"
df_test_raw = pd.read_parquet(file_path)

2.前処理

In [ ]:
# 外れ値対策
# 数値特徴量のクリッピングの閾値
# 企業の顧客DBなどの分布と合わせた方がいいと思うが、今回は公開データなので、学習データを基準とする

dict_feature_clipping_thresholds = {
    "age":{"top":59, "bottom":27}
    ,"balance":{"top":5768, "bottom":-172}
    ,"duration":{"top":751, "bottom":35}
    ,"campaign":{"top":8, "bottom":1}
    ,"pdays":{"top":370, "bottom":79}
    ,"previous":{"top":3,"bottom":0}
}

logistic_svm_col_names_one_hot_encoding = [
    "job", "marital", "education", "contact", "month", "poutcome"
]

logistic_svm_col_names_binary = [
    "default", "housing", "loan", "y"
]

In [ ]:
# svmとロジスティック回帰に合わせた前処理にする

def preprocessing(df):
    
    # onehotencoding
    # カテゴリー型に変更→onehotencdingの順番で実施

    for col in logistic_svm_col_names_one_hot_encoding:
        df[col] = df[col].astype("category")

    # スクリプト化や実運用する際は、trainにカラムを合わせる。
    df = pd.get_dummies(df, columns=logistic_svm_col_names_one_hot_encoding)

    # binary(yes or no)のカラムを0,1フラグに変換
    for col in logistic_svm_col_names_binary:
        df[col] = df[col].str.lower().map({"yes": 1, "no": 0})
    
    # print(df.head(5))

    # pdaysをコンタクトフラグ(-1)と経過日数に分ける
    # コンタクトをしていない＝-1、なので、-1に該当するかどうか別カラムに分ける
    df["pdays_contact_flag"] = df["pdays"].isin([-1])
    # -1を除いた最頻値に置き換える
    mode = df_train_raw["pdays"][~(df["pdays"]==-1)].mode()[0]
    # print(mode)
    df["pdays"] = np.where(df["pdays"] == -1, mode, df["pdays"])

    # クリッピング,　正規化、標準化
    standard_scaler = StandardScaler()
    minmax_scaler = MinMaxScaler(feature_range=(0, 1))

    for k in dict_feature_clipping_thresholds:
        # 上限
        df[k] = np.where(
            df[k] >= dict_feature_clipping_thresholds[k]["top"]
            ,dict_feature_clipping_thresholds[k]["top"]
            ,df[k]
        )

        # 下限
        df[k] = np.where(
            df[k] <= dict_feature_clipping_thresholds[k]["bottom"]
            ,dict_feature_clipping_thresholds[k]["bottom"]
            ,df[k]
        )
    
        df[k] = minmax_scaler.fit_transform(df[[k]])
        df[k] = standard_scaler.fit_transform(df[[k]])


    return df

In [ ]:
# preprocessing_logistic_svmの動作確認
df_tmp = df_train_raw.copy()
df_tmp = preprocessing(df=df_tmp)

In [ ]:
# 前処理
df_train = df_train_raw.copy()
df_test = df_test_raw.copy()

df_train = preprocessing(df=df_train)
df_test = preprocessing(df=df_test)

3.学習

In [ ]:
y_col = "y"

X_train = df_train.drop(y_col, axis=1)
x_cols = X_train.columns
y_train = df_train[y_col]


In [ ]:
# ベースモデルの定義
estimators = [
    ("lgb", lgb.LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1)),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", tol=1e-2, cache_size=1000)), 
    ("lr", LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000))
]

# svm→tolをデフォルトの1e-3からゆるめる。cache_sizeをデフォルトの200(MB)から増やす。

In [ ]:
# メタモデル (ロジステック回帰)
stacking_model_logreg = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(class_weight="balanced", random_state=42),
    stack_method="predict_proba",
    cv=5 # 交差検証の分割数
)

In [ ]:
# メタモデルの学習(ロジスティック回帰)
stacking_model_logreg.fit(X_train, y_train)

In [ ]:
# 重みの確認
stacking_model_logreg.final_estimator_.coef_

In [ ]:
# メタモデル (xgboost)

scale_pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

stacking_model_xgboost = StackingClassifier(
    estimators=estimators,
    final_estimator=xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42),
    stack_method="predict_proba",
    cv=5 # 交差検証の分割数。自動的に StratifiedKFold（層化交差検証）
)

In [ ]:
# メタモデルの学習(xgboost)
stacking_model_xgboost.fit(X_train, y_train)

In [ ]:
# メタモデルの保存
joblib.dump(stacking_model_logreg, "../models/stacking_model_logreg.joblib")
joblib.dump(stacking_model_xgboost, "../models/stacking_model_xgboost.joblib")


In [ ]:
# メタモデルで予測
y_col = "y"

X_test = df_test.drop(y_col, axis=1)
x_cols_test = X_test.columns
y_test = df_test[y_col]


In [ ]:
# stacking_model_logreg
# stacking_model_xgboost

y_pred_prob = stacking_model_logreg.predict_proba(X_test)
y_pred = [1 if row[1] >= 0.5 else 0 for row in y_pred_prob]

In [ ]:
y_pred_prob

In [ ]:
# 各種指標の確認

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"precision_score: {precision_score(y_test, y_pred, pos_label=1)}")
print(f"recall_score: {recall_score(y_test, y_pred, pos_label=1)}")
print(f"f1_score: {f1_score(y_test, y_pred, pos_label=1)}")

4.陽性判定の閾値調整

In [ ]:

# PR曲線も求める
precision, recall, _ = precision_recall_curve(y_test, [row[1] for row in y_pred_prob])
ap_score = average_precision_score(y_test, [row[1] for row in y_pred_prob])

path_pr =  "../data/tmp/pr_curve_lr.png"
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"PR curve (ap_score = {ap_score:.3f})")
plt.plot([0, 1], [0, 1], "k--") # 比較用にランダム予測モデルを仮定して、対角線をひき
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision_Recall_curve")
plt.legend(loc="lower right")
plt.savefig(path_pr)

In [ ]:
y_pred_prob

In [ ]:
# クラス分類の閾値を動かした時のf1_scoreの変動確認

list_x =[]
list_y = []

for i in np.arange(0.3, 0.7, 0.1):
    
    y_pred_meta_threshold_test = [1 if row[1] >= i else 0 for row in y_pred_prob]
    f1 = f1_score(y_test, y_pred_meta_threshold_test, pos_label=1)

    list_x.extend([i])
    list_y.extend([f1])

plt.plot(list_x, list_y)
plt.xlabel("values of threshold")
plt.ylabel("f1 score of Logistic Regression")
plt.title("threshold vs f1 score")
plt.show()


In [ ]:
df_f1 = pd.DataFrame()
df_f1["threshold"] = list_x
df_f1["f1"] = list_y

# f1が最大の行を取得
df_f1.loc[df_f1["f1"].idxmax()]